<a href="https://colab.research.google.com/github/PavloIeremeiew/BD_FIT_1-7m/blob/main/BD_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pyspark

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from google.colab import files
import os

In [4]:
# Ініціалізація Spark сесії
spark = SparkSession.builder.appName("Lab4_SparkSQL").getOrCreate()

In [8]:
# Завантаження даних
uploaded = files.upload()
file_name0 = list(uploaded.keys())[0] #"results.csv"
df_results = spark.read.csv(file_name0, header=True, inferSchema=True)
df_results.createOrReplaceTempView("matches")

Saving results.csv to results.csv


In [9]:
print("1. Кількість матчів, зіграних збірними Німеччини та України:")
spark.sql("""
    SELECT
        SUM(CASE WHEN home_team = 'Germany' OR away_team = 'Germany' THEN 1 ELSE 0 END) AS Germany_Matches,
        SUM(CASE WHEN home_team = 'Ukraine' OR away_team = 'Ukraine' THEN 1 ELSE 0 END) AS Ukraine_Matches
    FROM matches
""").show()

1. Кількість матчів, зіграних збірними Німеччини та України:
+---------------+---------------+
|Germany_Matches|Ukraine_Matches|
+---------------+---------------+
|           1010|            320|
+---------------+---------------+



In [10]:
print("2. Кількість матчів у Києві та Лондоні:")
spark.sql("""
    SELECT city, COUNT(*) as match_count
    FROM matches
    WHERE city IN ('Kyiv', 'London')
    GROUP BY city
""").show()

2. Кількість матчів у Києві та Лондоні:
+------+-----------+
|  city|match_count|
+------+-----------+
|London|        428|
|  Kyiv|         97|
+------+-----------+



In [11]:
print("3. Топ-10 команд, що забили найбільше голів вдома:")
spark.sql("""
    SELECT home_team, SUM(home_score) as total_home_goals
    FROM matches
    GROUP BY home_team
    ORDER BY total_home_goals DESC
    LIMIT 10
""").show()

3. Топ-10 команд, що забили найбільше голів вдома:
+-----------+----------------+
|  home_team|total_home_goals|
+-----------+----------------+
|     Brazil|            1484|
|    Germany|            1337|
|  Argentina|            1321|
|    England|            1230|
|     Sweden|            1207|
|     Mexico|            1179|
|    Hungary|            1118|
|South Korea|            1099|
|Netherlands|            1091|
|     France|            1080|
+-----------+----------------+



In [12]:
print("4. Топ-10 команд, що забили найбільше голів за весь час:")
spark.sql("""
    SELECT team, SUM(goals) as total_goals
    FROM (
        SELECT home_team as team, home_score as goals FROM matches
        UNION ALL
        SELECT away_team as team, away_score as goals FROM matches
    )
    GROUP BY team
    ORDER BY total_goals DESC
    LIMIT 10
""").show()

4. Топ-10 команд, що забили найбільше голів за весь час:
+-----------+-----------+
|       team|total_goals|
+-----------+-----------+
|    England|       2338|
|     Brazil|       2270|
|    Germany|       2257|
|     Sweden|       2117|
|    Hungary|       1984|
|  Argentina|       1981|
|Netherlands|       1773|
|South Korea|       1764|
|     Mexico|       1726|
|     France|       1649|
+-----------+-----------+



In [13]:
print("5. Середній бал (кількість голів) для кожної команди (Топ-10):")
spark.sql("""
    SELECT team, AVG(goals) as avg_goals
    FROM (
        SELECT home_team as team, home_score as goals FROM matches
        UNION ALL
        SELECT away_team as team, away_score as goals FROM matches
    )
    GROUP BY team
    ORDER BY avg_goals DESC
    LIMIT 10
""").show()

5. Середній бал (кількість голів) для кожної команди (Топ-10):
+------------------+------------------+
|              team|         avg_goals|
+------------------+------------------+
|       Elba Island|               5.0|
|         Yorkshire| 3.857142857142857|
|Parishes of Jersey|3.6666666666666665|
|          Cascadia|3.2857142857142856|
|         Occitania| 3.121212121212121|
|             Sápmi| 3.103448275862069|
|       Isle of Man|3.0754716981132075|
|           Silesia|               3.0|
|          Asturias|               3.0|
|            Surrey|               3.0|
+------------------+------------------+



частина 2

In [14]:
# Створення DataFrame 1 (Employees)
emp_data = [
    (1, "John", "IT"), (2, "Alice", "HR"),
    (3, "Bob", "IT"), (4, "Charlie", "Sales"),
    (5, "Diana", "Marketing")
]
df_emp = spark.createDataFrame(emp_data, ["employee_id", "name", "department"])
df_emp.createOrReplaceTempView("Employees")


# Створення DataFrame 2 (Projects)
proj_data = [
    (101, "Alpha", 1), (102, "Beta", 2),
    (103, "Gamma", 3), (104, "Delta", None),
    (105, "Epsilon", 6)
]
df_proj = spark.createDataFrame(proj_data, ["project_id", "project_name", "employee_id"])
df_proj.createOrReplaceTempView("Projects")

In [15]:
print("Inner Join (Тільки працівники відділу IT):")
inner_df = spark.sql("""
    SELECT e.*, p.project_id, p.project_name
    FROM Employees e
    INNER JOIN Projects p ON e.employee_id = p.employee_id
    WHERE e.department = 'IT'
""")
inner_df.show()
inner_df.write.mode("overwrite").parquet("inner_join_result.parquet")

Inner Join (Тільки працівники відділу IT):
+-----------+----+----------+----------+------------+
|employee_id|name|department|project_id|project_name|
+-----------+----+----------+----------+------------+
|          1|John|        IT|       101|       Alpha|
|          3| Bob|        IT|       103|       Gamma|
+-----------+----+----------+----------+------------+



In [16]:
print("Left Join (Підрахунок працівників у кожному відділі):")
left_df = spark.sql("""
    SELECT e.department, COUNT(e.employee_id) as total_employees
    FROM Employees e
    LEFT JOIN Projects p ON e.employee_id = p.employee_id
    GROUP BY e.department
""")
left_df.show()
left_df.write.mode("overwrite").parquet("left_join_result.parquet")

Left Join (Підрахунок працівників у кожному відділі):
+----------+---------------+
|department|total_employees|
+----------+---------------+
|     Sales|              1|
|        HR|              1|
| Marketing|              1|
|        IT|              2|
+----------+---------------+



In [17]:
print("Right Join (Усі проекти та залучені працівники):")
right_df = spark.sql("""
    SELECT p.project_id, p.project_name, e.name, e.department
    FROM Employees e
    RIGHT JOIN Projects p ON e.employee_id = p.employee_id
""")
right_df.show()
right_df.write.mode("overwrite").parquet("right_join_result.parquet")

Right Join (Усі проекти та залучені працівники):
+----------+------------+-----+----------+
|project_id|project_name| name|department|
+----------+------------+-----+----------+
|       101|       Alpha| John|        IT|
|       102|        Beta|Alice|        HR|
|       104|       Delta| NULL|      NULL|
|       105|     Epsilon| NULL|      NULL|
|       103|       Gamma|  Bob|        IT|
+----------+------------+-----+----------+



In [19]:
print("Full Outer Join:")
full_df = spark.sql("""
    SELECT *
    FROM Employees e
    FULL OUTER JOIN Projects p USING (employee_id)
""")
full_df.show()
full_df.write.mode("overwrite").parquet("full_join_result.parquet")

Full Outer Join:
+-----------+-------+----------+----------+------------+
|employee_id|   name|department|project_id|project_name|
+-----------+-------+----------+----------+------------+
|       NULL|   NULL|      NULL|       104|       Delta|
|          1|   John|        IT|       101|       Alpha|
|          2|  Alice|        HR|       102|        Beta|
|          3|    Bob|        IT|       103|       Gamma|
|          4|Charlie|     Sales|      NULL|        NULL|
|          5|  Diana| Marketing|      NULL|        NULL|
|          6|   NULL|      NULL|       105|     Epsilon|
+-----------+-------+----------+----------+------------+



In [21]:
print("Cross Join:")
cross_df = spark.sql("""
    SELECT e.*, p.project_id, p.project_name
    FROM Employees e
    CROSS JOIN Projects p
""")

cross_df.show(5)
cross_df.write.mode("overwrite").parquet("cross_join_result.parquet")

Cross Join:
+-----------+-----+----------+----------+------------+
|employee_id| name|department|project_id|project_name|
+-----------+-----+----------+----------+------------+
|          1| John|        IT|       101|       Alpha|
|          1| John|        IT|       102|        Beta|
|          2|Alice|        HR|       101|       Alpha|
|          2|Alice|        HR|       102|        Beta|
|          1| John|        IT|       103|       Gamma|
+-----------+-----+----------+----------+------------+
only showing top 5 rows


частина 3

In [50]:
#uploaded = files.upload()
#file_name1 = list(uploaded.keys())[0] #"review.json"

from google.colab import drive
drive.mount('/content/drive')

df_review = spark.read.json("/content/drive/MyDrive/Colab Notebooks/yelp_dataset/yelp_academic_dataset_review.json")
df_review.createOrReplaceTempView("review")

Mounted at /content/drive


In [47]:
uploaded = files.upload()
file_name2 = list(uploaded.keys())[0] #"business.json"

df_business = spark.read.json(file_name2)
df_business.createOrReplaceTempView("business")

Saving yelp_academic_dataset_business.json to yelp_academic_dataset_business.json


In [52]:
#1. 5 найкращих користувачів, які дали найбільше 5-зіркових відгуків
spark.sql("""
    SELECT user_id, stars, COUNT(*) as starcount
    FROM review
    GROUP BY user_id, stars
    HAVING stars = 5
    ORDER BY starcount DESC
    LIMIT 5
""").show(truncate=False)

+----------------------+-----+---------+
|user_id               |stars|starcount|
+----------------------+-----+---------+
|vHc-UrI9yfL_pnnc6nJtyQ|5.0  |655      |
|Xw7ZjaGfr0WNVt6s_5KZfA|5.0  |635      |
|ET8n-r7glWYqZhuR6GcdNw|5.0  |583      |
|pou3BbKsIozfH50rxmnMew|5.0  |568      |
|AHRrG3T1gJpHvtpZ-K0G_g|5.0  |554      |
+----------------------+-----+---------+



In [53]:
# 2. Відсоток загальної кількості відгуків із зірочками S
spark.sql("""
    WITH user_stars AS (
        SELECT user_id, stars, COUNT(*) as starcount
        FROM review
        GROUP BY user_id, stars
    ),
    total_stars AS (
        SELECT stars, SUM(starcount) as total
        FROM user_stars
        GROUP BY stars
    )
    SELECT u.user_id, u.stars, u.starcount, t.total,
           (u.starcount / t.total) * 100 as percent
    FROM user_stars u
    JOIN total_stars t ON u.stars = t.stars
    ORDER BY u.user_id LIMIT 10
""").show()

+--------------------+-----+---------+-------+--------------------+
|             user_id|stars|starcount|  total|             percent|
+--------------------+-----+---------+-------+--------------------+
|---1lKK3aKOuomHnw...|  5.0|        1|3231627|3.094416527650004E-5|
|---2PmXbF47D870st...|  5.0|       28|3231627|8.664366277420012E-4|
|---UgP94gokyCDuB5...|  3.0|        1| 691934|1.445224544537485E-4|
|---UgP94gokyCDuB5...|  1.0|        2|1069561|1.869926072472725E-4|
|---UgP94gokyCDuB5...|  4.0|        5|1452918|3.441350440974646E-4|
|---UgP94gokyCDuB5...|  5.0|        3|3231627|9.283249582950012E-5|
|---fa6ZK37T9NjkGK...|  1.0|        1|1069561|9.349630362363624E-5|
|---r61b7EpVPkb4UV...|  5.0|        3|3231627|9.283249582950012E-5|
|---r61b7EpVPkb4UV...|  4.0|        1|1452918|6.882700881949292E-5|
|---r61b7EpVPkb4UV...|  1.0|        1|1069561|9.349630362363624E-5|
+--------------------+-----+---------+-------+--------------------+



In [54]:
# 3. Середня кількість зірок для компаній з їжею на винос (RestaurantsTakeOut)
spark.sql("""
    SELECT AVG(stars) as avg_stars
    FROM business
    WHERE attributes.RestaurantsTakeOut = 'True'
""").show()

+------------------+
|         avg_stars|
+------------------+
|3.5336116200441983|
+------------------+



In [55]:
# 4. 5 найкращих користувачів, які пишуть відгуки з найбільшою середньою довжиною слів
spark.sql("""
    SELECT user_id, AVG(size(split(text, ' '))) as avgW
    FROM review
    GROUP BY user_id
    ORDER BY avgW DESC
    LIMIT 5
""").show()

+--------------------+------+
|             user_id|  avgW|
+--------------------+------+
|30Y0D93d9rypoql_X...|1295.0|
|USKesOeJ5myMUJl6N...|1134.0|
|PHNLJlUAAOG7hNYeE...|1080.0|
|1-bR3vOU7YyKwvm2N...|1067.0|
|mnB9mS00SY_udD0cK...|1056.0|
+--------------------+------+



частина 4

In [22]:
uploaded = files.upload()
file_name3 = list(uploaded.keys())[0]

# 2. Завантаження власного датасету (Відеоігри)
df_games = spark.read.csv(file_name3, header=True, inferSchema=True)

# 3. Реєстрація DataFrame як тимчасового SQL подання
df_games.createOrReplaceTempView("games")

# Перевірка схеми
df_games.printSchema()

Saving власний-датасет.csv to власний-датасет.csv
root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)
 |-- Critic_Score: string (nullable = true)
 |-- Critic_Count: string (nullable = true)
 |-- User_Score: string (nullable = true)
 |-- User_Count: string (nullable = true)
 |-- Developer: string (nullable = true)
 |-- Rating: string (nullable = true)



In [33]:
print("1. Топ-10 найпопулярніших платформ за кількістю випущених ігор")
# Демонструє групування (GROUP BY) та сортування (ORDER BY)
spark.sql("""
    SELECT Platform, COUNT(Name) as total_games
    FROM games
    GROUP BY Platform
    ORDER BY total_games DESC
    LIMIT 10
""").show()

1. Топ-10 найпопулярніших платформ за кількістю випущених ігор
+--------+-----------+
|Platform|total_games|
+--------+-----------+
|      DS|       2163|
|     PS2|       2161|
|     PS3|       1329|
|     Wii|       1325|
|    X360|       1265|
|     PSP|       1213|
|      PS|       1196|
|      PC|        960|
|      XB|        824|
|     GBA|        822|
+--------+-----------+



In [34]:
print("2. Топ-5 видавців (Publisher) за глобальними продажами")
# Демонструє агрегацію суми (SUM) та форматування чисел (ROUND)
spark.sql("""
    SELECT Publisher, ROUND(SUM(Global_Sales), 2) as Total_Global_Sales_Millions
    FROM games
    WHERE Publisher IS NOT NULL
    GROUP BY Publisher
    ORDER BY Total_Global_Sales_Millions DESC
    LIMIT 5
""").show(truncate=False)

2. Топ-5 видавців (Publisher) за глобальними продажами
+---------------------------+---------------------------+
|Publisher                  |Total_Global_Sales_Millions|
+---------------------------+---------------------------+
|Nintendo                   |1786.56                    |
|Electronic Arts            |1110.32                    |
|Activision                 |727.46                     |
|Sony Computer Entertainment|607.5                      |
|Ubisoft                    |474.72                     |
+---------------------------+---------------------------+



In [35]:
print("3. Динаміка глобальних продажів ігор по роках (Топ-10 найкращих років)")
# Аналіз часових трендів
spark.sql("""
    SELECT Year, ROUND(SUM(Global_Sales), 2) as Total_Sales
    FROM games
    WHERE Year IS NOT NULL
    GROUP BY Year
    ORDER BY Total_Sales DESC
    LIMIT 10
""").show()

3. Динаміка глобальних продажів ігор по роках (Топ-10 найкращих років)
+----+-----------+
|Year|Total_Sales|
+----+-----------+
|2008|      678.9|
|2009|      667.3|
|2007|     611.13|
|2010|     600.45|
|2006|     521.04|
|2011|     515.99|
|2005|     459.94|
|2004|     419.31|
|2002|     395.52|
|2013|     368.11|
+----+-----------+



In [36]:
print("4. Порівняння продажів у Північній Америці (NA) та Європі (EU) за жанрами")
# Демонструє роботу з кількома агрегатними функціями одночасно
spark.sql("""
    SELECT Genre,
           ROUND(SUM(NA_Sales), 2) as Total_NA_Sales,
           ROUND(SUM(EU_Sales), 2) as Total_EU_Sales
    FROM games
    GROUP BY Genre
    ORDER BY Total_NA_Sales DESC
""").show()

4. Порівняння продажів у Північній Америці (NA) та Європі (EU) за жанрами
+------------+--------------+--------------+
|       Genre|Total_NA_Sales|Total_EU_Sales|
+------------+--------------+--------------+
|      Action|        877.83|         525.0|
|      Sports|        683.35|        376.85|
|     Shooter|         582.6|        313.27|
|    Platform|        447.05|        201.63|
|        Misc|        410.24|        215.98|
|      Racing|        359.42|        238.39|
|Role-Playing|        327.28|        188.06|
|    Fighting|        223.59|        101.32|
|  Simulation|        183.31|        113.38|
|      Puzzle|        123.78|         50.78|
|   Adventure|         105.8|         64.13|
|    Strategy|          68.7|         45.34|
+------------+--------------+--------------+



In [37]:
print("5. Середня оцінка критиків для топ-5 жанрів (де є більше 100 відгуків)")
# Демонструє використання HAVING для фільтрації після групування та AVG
spark.sql("""
    SELECT Genre,
           ROUND(AVG(CAST(Critic_Score AS FLOAT)), 1) as Avg_Critic_Score,
           COUNT(*) as Games_With_Score
    FROM games
    WHERE Critic_Score IS NOT NULL
      AND Critic_Score NOT IN ('N/A', 'tbd', '')
    GROUP BY Genre
    HAVING COUNT(*) > 100
    ORDER BY Avg_Critic_Score DESC
    LIMIT 5
""").show()

5. Середня оцінка критиків для топ-5 жанрів (де є більше 100 відгуків)
+------------+----------------+----------------+
|       Genre|Avg_Critic_Score|Games_With_Score|
+------------+----------------+----------------+
|      Sports|            73.4|            1309|
|Role-Playing|            73.0|             757|
|    Strategy|            72.2|             295|
|     Shooter|            70.7|             948|
|    Fighting|            69.9|             435|
+------------+----------------+----------------+



In [38]:
print("6. Топ-10 ігор Nintendo з найвищими продажами в Японії (JP)")
# Демонструє просту фільтрацію за допомогою WHERE
spark.sql("""
    SELECT Name, Platform, JP_Sales
    FROM games
    WHERE Publisher = 'Nintendo'
    ORDER BY JP_Sales DESC
    LIMIT 10
""").show(truncate=False)

6. Топ-10 ігор Nintendo з найвищими продажами в Японії (JP)
+-------------------------------------------+--------+--------+
|Name                                       |Platform|JP_Sales|
+-------------------------------------------+--------+--------+
|Pokemon Red/Pokemon Blue                   |GB      |10.22   |
|Pokemon Gold/Pokemon Silver                |GB      |7.2     |
|Super Mario Bros.                          |NES     |6.81    |
|New Super Mario Bros.                      |DS      |6.5     |
|Pokemon Diamond/Pokemon Pearl              |DS      |6.04    |
|Pokemon Black/Pokemon White                |DS      |5.65    |
|Pokemon Ruby/Pokemon Sapphire              |GBA     |5.38    |
|Animal Crossing: Wild World                |DS      |5.33    |
|Brain Age 2: More Training in Minutes a Day|DS      |5.32    |
|New Super Mario Bros. Wii                  |Wii     |4.7     |
+-------------------------------------------+--------+--------+



In [39]:
print("7. Знаходження найуспішнішої гри для кожної платформи (Віконна функція)")
# Використання ROW_NUMBER() OVER (PARTITION BY ...)
spark.sql("""
    WITH RankedGames AS (
        SELECT Platform, Name, Global_Sales,
               ROW_NUMBER() OVER (PARTITION BY Platform ORDER BY Global_Sales DESC) as rank
        FROM games
        WHERE Global_Sales > 0
    )
    SELECT Platform, Name, Global_Sales
    FROM RankedGames
    WHERE rank = 1
    ORDER BY Global_Sales DESC
    LIMIT 10
""").show(truncate=False)

7. Знаходження найуспішнішої гри для кожної платформи (Віконна функція)
+--------+-----------------------------+------------+
|Platform|Name                         |Global_Sales|
+--------+-----------------------------+------------+
|Wii     |Wii Sports                   |82.74       |
|NES     |Super Mario Bros.            |40.24       |
|GB      |Pokemon Red/Pokemon Blue     |31.37       |
|DS      |New Super Mario Bros.        |30.01       |
|X360    |Kinect Adventures!           |21.82       |
|PS3     |Grand Theft Auto V           |21.4        |
|PS2     |Grand Theft Auto: San Andreas|20.81       |
|SNES    |Super Mario World            |20.61       |
|GBA     |Pokemon Ruby/Pokemon Sapphire|15.85       |
|3DS     |Pokemon X/Pokemon Y          |14.35       |
+--------+-----------------------------+------------+



In [40]:
print("8. Співвідношення оцінок критиків та гравців (для ігор з > 10 млн продажами)")
# Математичні операції всередині запиту (приведення 100-бальної системи до 10-бальної)
spark.sql("""
    SELECT Name,
           Global_Sales,
           Critic_Score,
           CAST(User_Score AS FLOAT) * 10 as User_Score_100_Scale
    FROM games
    WHERE Global_Sales >= 10
      AND Critic_Score IS NOT NULL
      AND User_Score NOT IN ('tbd', 'N/A')
    ORDER BY Global_Sales DESC
    LIMIT 10
""").show(truncate=False)

8. Співвідношення оцінок критиків та гравців (для ігор з > 10 млн продажами)
+-------------------------+------------+------------+--------------------+
|Name                     |Global_Sales|Critic_Score|User_Score_100_Scale|
+-------------------------+------------+------------+--------------------+
|Wii Sports               |82.74       |76          |80.0                |
|Mario Kart Wii           |35.82       |82          |83.00000190734863   |
|Wii Sports Resort        |33.0        |80          |80.0                |
|New Super Mario Bros.    |30.01       |89          |85.0                |
|Wii Play                 |29.02       |58          |65.99999904632568   |
|New Super Mario Bros. Wii|28.62       |87          |83.99999618530273   |
|Mario Kart DS            |23.42       |91          |86.00000381469727   |
|Wii Fit                  |22.72       |80          |76.99999809265137   |
|Wii Fit Plus             |22.0        |80          |74.00000095367432   |
|Kinect Adventures!    